# Project: RAG - Q&A Application on Your Private Documents (Pinecone & Chroma)

Question answering Pipeline
1. Prepare the document (once per document)
- Load data in Langchain Documents
- split the document into chunks
- embed the chunks into numeric vectors
- save the chunks and embeddings to a vector database

2. Search(once per query)
- embed the user's questions
- Use the question's embedding and chunk embeddings, ranking the vectors by similarity

3. Ask(once per query)
- Insert the question and the most relevant chunks intoa message to GPT model
- Return GPT's answer

In [ ]:
!pip install -r "/content/drive/MyDrive/Colab Notebooks/Langchain_Project/requirements.txt"

In [ ]:
pip install -q pinecone

In [ ]:
pip show pinecone

In [ ]:
import os

os.environ["PINECONE_API_KEY"] = "openai_api"
os.environ["OPENAI_API_KEY"] = "sk-proj-openai_api"

from google.colab import userdata
userdata.get('openai_api')
userdata.get('pinecone_api')



In [ ]:
from pinecone import Pinecone

pc = Pinecone(api_key=pinecone_api_key)



In [ ]:
pip install pypdf -q

In [ ]:
pip install docx2txt -q

In [ ]:
pip install wikipedia -q

# Loading the documents

Create a function to load private and public documents

In [ ]:
def load_document(file):
    import os
    name, extension = os.path.splitext(file)

    if extension == '.pdf':
        from langchain.document_loaders import PyPDFLoader
        print(f'Loading {file}')
        loader = PyPDFLoader(file)
    elif extension == '.docx':
        from langchain.document_loaders import Docx2txtLoader
        print(f'Loading {file}')
        loader = Docx2txtLoader(file)
    else:
        raise ValueError(f"Unsupported file type: {extension}")

    data = loader.load()
    return data

def load_from_wikipedia(query, lang = 'en', load_max_docs = 2):
    from langchain.document_loaders import WikipediaLoader
    loader = WikipediaLoader(query=query, lang=lang, load_max_docs=load_max_docs)
    data = loader.load()
    return data

# Chunking Data

In [ ]:
def chunk_data(data, chunk_size = 256):
    from langchain.text_splitter import RecursiveCharacterTextSplitter
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=0)
    chunks = text_splitter.split_documents(data)
    return chunks

# Calculating Cost

In [ ]:
def print_embedding_cost(texts):
    import tiktoken
    enc = tiktoken.encoding_for_model('text-embedding-3-small')
    total_tokens = sum([len(enc.encode(page.page_content)) for page in texts])
    print(f'Total Tokens: {total_tokens}')
    print(f'Embedding Cost in USD: {total_tokens / 1000 * 0.00002:.6f}')

In [ ]:
pip install langchain-pinecone


# Embedding and Uploading to a Vector Database(PineCone)

In [ ]:
%pip install --upgrade langchain langchain-pinecone pinecone-client tiktoken


In [ ]:
def insert_or_fetch_embeddings(index_name, chunks):
    # importing the necessary libraries and initializing the Pinecone client
    import pinecone
    from langchain_community.vectorstores import Pinecone
    from langchain_openai import OpenAIEmbeddings
    from pinecone import ServerlessSpec
    from langchain_pinecone import PineconeVectorStore



    pc = pinecone.Pinecone()

    embeddings = OpenAIEmbeddings(model='text-embedding-3-small', dimensions=1536)  # 512 works as well

    # loading from existing index
    if index_name in pc.list_indexes().names():
        print(f'Index {index_name} already exists. Loading embeddings ... ', end='')
        # Use Pinecone class from langchain_community.vectorstores
        vector_store = Pinecone.from_existing_index(index_name, embeddings)
        print('Ok')
    else:
        # creating the index and embedding the chunks into the index
        print(f'Creating index {index_name} and embeddings ...', end='')

        # creating a new index
        pc.create_index(
            name=index_name,
            dimension=1536,
            metric='cosine',
            spec=ServerlessSpec(
                cloud="aws",
                region="us-east-1"
        )
        )

        # processing the input documents, generating embeddings using the provided `OpenAIEmbeddings` instance,
        # inserting the embeddings into the index and returning a new Pinecone vector store object.
        vector_store = PineconeVectorStore.from_documents(
          documents=chunks,      # your text chunks
          embedding=embeddings,   # your embedding model
          index_name=index_name   # your existing Pinecone index
          )
        print('Ok')

    return vector_store

In [ ]:
from pinecone import Pinecone

def delete_pinecone_index(index_name="all"):
    pc = Pinecone()  # uses PINECONE_API_KEY from env by default

    if index_name == "all":
        # list_indexes returns IndexList object with .names()
        for name in pc.list_indexes().names():
            print(f"Deleting index: {name}")
            pc.delete_index(name)
    else:
        print(f"Deleting index: {index_name}")
        pc.delete_index(index_name)




# Asking and Getting Answers

In [ ]:
def ask_and_get_answer(vector_store, q):

  from langchain.chains import RetrievalQA
  from langchain_openai import ChatOpenAI

  # Initialize the LLM with the specified model and temperature
  llm = ChatOpenAI(model='gpt-3.5-turbo', temperature=0.2)

  # Use the provided vector store with similarity search and retrieve top 3 results
  retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={'k': 3})

  # Create a RetrievalQA chain using the defined LLM, chain type 'stuff', and retriever
  chain = RetrievalQA.from_chain_type(llm=llm, chain_type='stuff', retriever=retriever)

  answer = chain.run(q)
  return answer

# Running the code

In [ ]:
data = load_document('/content/drive/MyDrive/Colab Notebooks/Langchain_Project/files/us_constitution.pdf')
#data = load_document('files/the_great_gatsby.docx')
#data = load_from_wikipedia('GPT-4')
print(data[0].page_content)

In [ ]:
print(f"You have {len(data)} pages in your document.")
print(f"The first page has {len(data[0].page_content)} characters.")

In [ ]:
chunks = chunk_data(data)
print(len(chunks))
print(chunks[10].page_content)

In [ ]:
print_embedding_cost(chunks)

In [ ]:
delete_pinecone_index()

In [ ]:
index_name = 'askdocument'
vector_store = insert_or_fetch_embeddings(index_name,chunks)

In [ ]:
q = "What is the whole document about?"
answer = ask_and_get_answer(vector_store, q)
print(answer)

In [ ]:
# Creating a loop so the user can ask questions
import time
i=1
print('Write Quit or Exit or quit')
while True:
  q = input(f'Question #  {i} : ')
  i = i + 1
  if q.lower() in ['quit','exit']:
    print('Quitting')
    time.sleep(2)
    break
  answer = ask_and_get_answer(vector_store, q)
  print(f'Answer: {answer}')
  print(f'\n {"-" * 50} \n')

In [ ]:
delete_pinecone_index()

In [ ]:
data = load_from_wikipedia('ChatGPT', 'ro')
chunks = chunk_data(data)
index_name = 'chatgpt'
vector_store = insert_or_fetch_embeddings(index_name, chunks)

In [ ]:
q = "what is chatgpt"
answer = ask_and_get_answer(vector_store, q)
print(answer)

# Using Chroma as a Vector DB

In [ ]:
pip install -q chromadb

In [ ]:
def create_embeddings_chroma(chunks, persistant_directory='./chroma_db'):
    from langchain.vectorstores import Chroma
    from langchain_openai import OpenAIEmbeddings

    embeddings = OpenAIEmbeddings(model='text-embedding-3-small', dimensions=1536)
    vector_store = Chroma.from_documents(chunks, embeddings, persist_directory=persistant_directory)
    return vector_store

def load_embeddings_chroma(persistant_directory='./chroma_db'):
    from langchain.vectorstores import Chroma
    from langchain_openai import OpenAIEmbeddings

    embeddings = OpenAIEmbeddings(model='text-embedding-3-small', dimensions=1536)
    vector_store = Chroma(persist_directory=persistant_directory, embedding_function=embeddings)
    return vector_store


In [ ]:
data = load_document('/content/drive/MyDrive/Colab Notebooks/Langchain_Project/files/rag_powered_by_google_search.pdf')
chunks = chunk_data(data, chunk_size = 256)
vector_store = create_embeddings_chroma(chunks)

In [ ]:
q = 'What is Vertix Ai search'
answer = ask_and_get_answer(vector_store, q)
print(answer)

In [ ]:
db = load_embeddings_chroma()
q = 'How many pairs of questions and answer had the StackOverflow Dtataset?'
answer = ask_and_get_answer(db, q)
print(answer)

In [ ]:
q = 'Mutip'

# Adding Memory (chat history)

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.chains import ConversationalRetrievalChain  # Import class for building conversational AI chains
from langchain.memory import ConversationBufferMemory  # Import memory for storing conversation history

# Instantiate a ChatGPT LLM (temperature controls randomness)
llm = ChatOpenAI(model_name='gpt-3.5-turbo', temperature=0)

# Configure vector store to act as a retriever (finding similar items, returning top 5)
retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={'k': 5})


# Create a memory buffer to track the conversation
memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)

crc = ConversationalRetrievalChain.from_llm(
    llm=llm,  # Link the ChatGPT LLM
    retriever=retriever,  # Link the vector store based retriever
    memory=memory,  # Link the conversation memory
    chain_type='stuff',  # Specify the chain type
    verbose=False  # Set to True to enable verbose logging for debugging
)


In [ ]:
# function to ask questions
def ask_question(q, chain):
    result = chain.invoke({'question': q})
    return result

In [ ]:
data = load_document('/content/drive/MyDrive/Colab Notebooks/Langchain_Project/files/rag_powered_by_google_search.pdf')
chunks = chunk_data(data, chunk_size=256)
vector_store = create_embeddings_chroma(chunks)

In [ ]:
q = 'How many pairs of questions and answers had the StackOverflow dataset?'
result = ask_question(q, crc)
print(result)

In [ ]:
print(result['answer'])

In [ ]:
q = 'Mutiply that number by 10'
result = ask_question(q, crc)
print(result)

In [ ]:
for item in result['chat_history']:
    print(item)

# Looping

In [ ]:
# Creating a loop so the user can ask questions
import time
i=1
print('Write Quit or Exit or quit')
while True:
  q = input(f'Question #  {i} : ')
  i = i + 1
  if q.lower() in ['quit','exit']:
    print('Quitting')
    time.sleep(2)
    break
  answer = ask_and_get_answer(vector_store, q)
  print(f'Answer: {answer}')
  print(f'\n {"-" * 50} \n')

# Using Custom Prompt

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory

from langchain.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate

llm = ChatOpenAI(model_name='gpt-3.5-turbo', temperature=0)
retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={'k': 5})
memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)


system_template = r'''
Use the following pieces of context to answer the user's question.
Before answering translate your response to French.
If you don't find the answer in the provided context, just respond "I don't know."
---------------
Context: ```{context}```
'''

user_template = '''
Question: ```{question}```
'''

messages= [
    SystemMessagePromptTemplate.from_template(system_template),
    HumanMessagePromptTemplate.from_template(user_template)
]

qa_prompt = ChatPromptTemplate.from_messages(messages)

crc = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    chain_type='stuff',
    combine_docs_chain_kwargs={'prompt': qa_prompt },
    verbose=True
)

In [ ]:
print(qa_prompt)

In [ ]:
db = load_embeddings_chroma()
q = 'How many pairs of questions and answers had the StackOverflow dataset?'
result = ask_question(q, crc)
print(result)

In [ ]:
q = 'When was elon musk born?'
result = ask_question(q, crc)
print(result)